### This notebook includes example code to run the analyses shown in the manuscript:

*PLSpy: An Open-Source Python Implementation of Partial Least Squares (Rokos et al., 2026)*

---
&nbsp;

**To run this example**, the simulated data file will be available for download from Zenodo at: [10.5281/zenodo.24464600](https://doi.org/10.5281/zenodo.24464600)

&nbsp;

The first cell below:
- Loads the simulated data (`X`, `Y`)
- Runs a behavioural PLS
- Saves the full PLS results to a pickle file

**Parameters:**
- `groups_sizes`: number of rows (e.g., participants) for each group (in this example: 1 group of 10,000)
- `num_conditions`: number of conditions
- `num_perm`: number of iterations for permutation testing
- `num_boot`: number of iterations for bootstrap resampling
- `pls_method`: 'rb' = behavioural PLS
- `num_split`: number of iterations for split-half resampling
- `lv`: number of latent variables to examine with split-half resampling
- `CI`: confidence interval for split-half resampling

In [ ]:
import plspy
import numpy as np
import scipy.io
import pickle

# ---- LOAD SIMULATION DATA ----
simdata = scipy.io.loadmat('/path/to/data-simrel.mat')
X = simdata['X']
Y = simdata['Y']

# ---- RUN PLS ANALYSIS ----
sim_result = plspy.PLS(X, groups_sizes=[10000], num_conditions=1, Y=Y, num_perm = 2000, num_boot=2000, pls_method = "rb", num_split = 2000, lv =4, CI = 0.95)

# ---- SAVE RESULTS ----
with open("simrel_pls-results.pkl", "wb") as f:
    pickle.dump(sim_result, f)

---
&nbsp;
### Generate PLS Summary Table

- Extracts key outputs from the PLS results generated above (`simrel_pls-results.pkl`)  
- Prepares a summary table with:
  - Singular values
  - Permutation p-values  
  - Split-half z-scores and respective null z-scores 
  - Split-half Test–train z-scores and respective null z-scores 
- Prints the table for inspection  
- Saves the summary as a TSV file:  
  **`simrel_pls-results_summary.tsv`**

In [ ]:
import pickle
import pandas as pd

# ---- LOAD PLS RESULT ----
with open("simrel_pls-results.pkl", "rb") as f:
    r = pickle.load(f)

# ---- PREPARE DATAFRAME ----
# Extract key outputs from the PLS object and organize into columns:
df = pd.DataFrame({
    "LV": [f"LV{i+1}" for i in range(len(r.s))],
    "Singular Value": r.s,
    "Singular Value p": r.resample_tests.permute_ratio,
    "Step-down p": r.resample_tests.stepdown_ratio,
    "Split-half z u": r.pls_repro_sh['pls_rep_z_u'],
    "Split-half z v": r.pls_repro_sh['pls_rep_z_v'],
    "Split-half null z u": r.pls_repro_sh['pls_null_z_u'],
    "Split-half null z v": r.pls_repro_sh['pls_null_z_v'],
    "Test–train z": r.pls_repro_tt['z'],
    "Test–train null z": r.pls_repro_tt['z_null']
}).set_index("LV")

# ---- PRINT TABLE OF RESULTS ----
def print_table(df, decimals=3):
    df_print = df.round(decimals)
    print(df_print.to_string())

print_table(df)

# ---- SAVE TSV ----
df.to_csv("simrel_pls-results_summary.tsv", sep="\t", float_format="%.6f")
print("\nSaved outputs to simrel_pls-results_summary.tsv")

---
&nbsp;
### Plot Singular Vector Values for Each Latent Variable

- Loads the PLS results generated above (`simrel_pls-results.pkl`)  
- Extracts the singular vector (`U`) and bootstrap ratios (`BSRs`)  
- Plots the singular vector (`U`) for each latent variable (LV)  
- Highlights the features with significant bootstrap ratios (`|BSR| >= 2`) in red  
 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle

# ---- LOAD PLS RESULT ----
with open("simrel_pls-results.pkl", "rb") as f:
    sim_result = pickle.load(f)

# ---- EXTRACT SINGULAR VECTORS AND BSRs ----
U = sim_result.U
bsr = sim_result.resample_tests.boot_ratios  # BSRs
num_LVs = U.shape[1]
num_features = U.shape[0]

# ---- CREATE FIGURE ----
# Stacked bar plots: one subplot per LV
fig, axes = plt.subplots(
    num_LVs, 1,
    figsize=(6, 15),  
    sharex=True
)
# ---- PLOT EACH LV ----
for lv in range(num_LVs):
    ax = axes[lv] if num_LVs > 1 else axes

    # Extract weights for this LV
    weights = U[:, lv]
    
    # Highlight |BSR| >= 2
    sig = np.abs(bsr[:, lv]) >= 2
    colors = ["red" if s else "lightgrey" for s in sig]

    # Grey reference line at zero
    ax.axhline(y=0, color="grey", linewidth=0.6, zorder=2)
    
    # Plot bar chart
    ax.bar(np.arange(num_features), weights, color=colors)
    ax.grid(False)

    # Customize ticks
    ax.tick_params(axis="both", labelsize=12, width=0.2, length=2)
    
    # Customize subplot borders
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_color("grey")
    ax.spines["left"].set_color("grey")
    ax.tick_params(axis="y", left=False)
    axes[-1].set_xlabel("Features", fontsize=12, fontweight="bold",labelpad=2)

    # Add title only to the first subplot
    if lv == 0:
        ax.set_title("U", fontsize=14, fontweight="bold", pad=3)

# ---- DISPLAY FIGURE & SAVE ----

fig.subplots_adjust(
    left=0.10,
    right=0.99,
    top=0.94,
    bottom=0.08,
    hspace=0.25
)

plt.savefig(
    "simrel_pls_U.png",
    dpi=600,
    bbox_inches="tight",
    format="png"
)

plt.show()

---
&nbsp;
### Plot Correlation Values for Each Latent Variable

- Loads the PLS results generated above (`simrel_pls-results.pkl`)  
- Uses `plspy.visualize` to plot correlation values for each latent variable (LV)  

In [ ]:
# Plot the correlation values for each LV
from plspy.visualize import visualize
import pickle

# ---- LOAD PLS RESULT ----
with open("simrel_pls-results.pkl", "rb") as f:
    sim_result = pickle.load(f)

# Create plots for all LVs
for lv in range(sim_result.s.shape[0]):
    visualize.visualize(plot="cor", pls_result=sim_result, lv=lv)